In [29]:
import numpy as np
import pandas as pd
from typing import Optional, Tuple, Dict, Any, List

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, cross_val_score
import json
import pickle

In [21]:
def load_sid_split(in_json):
    """
    Load train/test subject IDs from a JSON.
    """
    with open(in_json, "r", encoding="utf-8") as f:
        obj = json.load(f)
    train = set(obj["train_sids"])
    test  = set(obj["test_sids"])
    if train & test:
        raise ValueError("Loaded split has overlapping sids.")
    return train, test, obj.get("meta", {})

# --- APPLY TO ANY DATAFRAME ---

def apply_sid_split(data, train_sids, test_sids, sid_col="sid", sex_col="sex"):
    """
    Given a DataFrame and saved subject IDs, return aligned splits for all/male/female.
    """
    sid_as_str = data[sid_col].astype(str)
    is_train = sid_as_str.isin(train_sids)
    is_test  = sid_as_str.isin(test_sids)

    train_all = data[is_train].copy()
    test_all  = data[is_test].copy()

    male   = data[data[sex_col] == "M"]
    female = data[data[sex_col] == "F"]

    train_m = male[male[sid_col].astype(str).isin(train_sids)].copy()
    test_m  = male[male[sid_col].astype(str).isin(test_sids)].copy()
    train_f = female[female[sid_col].astype(str).isin(train_sids)].copy()
    test_f  = female[female[sid_col].astype(str).isin(test_sids)].copy()

    return {"all": (train_all, test_all),
            "male": (train_m, test_m),
            "female": (train_f, test_f)}

In [22]:
def build_nextx_dataset(
    df: pd.DataFrame,
    use_dt: bool = True,
    hop: int = 1,
    sid_col: str = "sid",
    age_col: str = "age",
    x_col: str = "x",
    y_col: str = "y",
    landmark_col: Optional[str] = "landmark",
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    """
    Build (t -> t+hop) pairs jointly for x,y.
      X = [x_t, y_t, (dt)],   Y = [x_{t+hop}, y_{t+hop}]
      groups = sid  (for GroupKFold)
      meta: sid, (landmark?), age_t, age_t+hop, dt
    """
    X_rows: List[List[float]] = []
    Y_rows: List[List[float]] = []
    G_rows: List[Any] = []
    meta_rows: List[Dict[str, Any]] = []

    if landmark_col is None:
        grouped = df.groupby(sid_col, sort=False)
    else:
        grouped = df.groupby([sid_col, landmark_col], sort=False)

    for _, g in grouped:
        g = g[[sid_col, age_col, x_col, y_col] + ([landmark_col] if landmark_col else [])].dropna()
        if g.empty:
            continue
        g = g.sort_values(age_col)

        ages = g[age_col].to_numpy(float)
        xs   = g[x_col].to_numpy(float)
        ys   = g[y_col].to_numpy(float)

        n = len(xs)
        if n <= hop:
            continue

        for i in range(n - hop):
            xt, yt = xs[i], ys[i]
            x_next, y_next = xs[i + hop], ys[i + hop]
            dt_val = float(ages[i + hop] - ages[i])

            feat = [xt, yt]
            if use_dt:
                feat.append(dt_val)

            X_rows.append(feat)
            Y_rows.append([x_next, y_next])
            sid_val = g[sid_col].iloc[0]
            if landmark_col:
                lm_val = g[landmark_col].iloc[0]
                meta_rows.append({
                    "sid": sid_val, landmark_col: lm_val,
                    "age_t": ages[i], f"age_t+{hop}": ages[i + hop], "dt": dt_val,
                })
            else:
                meta_rows.append({
                    "sid": sid_val,
                    "age_t": ages[i], f"age_t+{hop}": ages[i + hop], "dt": dt_val,
                })
            G_rows.append(sid_val)

    if not X_rows:
        # sane empty returns
        d = 3 if use_dt else 2
        return (np.zeros((0, d), float),
                np.zeros((0, 2), float),
                np.zeros((0,), object),
                pd.DataFrame(columns=["sid", "age_t", f"age_t+{hop}", "dt"] + ([landmark_col] if landmark_col else []))
               )

    X = np.asarray(X_rows, float)
    Y = np.asarray(Y_rows, float)
    groups = np.asarray(G_rows, object)
    meta = pd.DataFrame(meta_rows)
    return X, Y, groups, meta


def fit_nextx_regressor(
    X: np.ndarray,
    Y: np.ndarray,
    groups: np.ndarray,
    *,
    alpha: float = 1.0,
    cv_splits: int = 5,
    poly_degree: int = 1,   # 1=Linear Ridge; >1=Poly-Ridge
    random_state: int = 0,
) -> Tuple[Pipeline, Dict[str, Any]]:
    steps = []
    if poly_degree and poly_degree > 1:
        steps.append(("poly", PolynomialFeatures(degree=poly_degree, include_bias=False)))
    steps.append(("scaler", StandardScaler()))
    steps.append(("ridge", Ridge(alpha=alpha, random_state=random_state)))
    pipe = Pipeline(steps)

    uniq = np.unique(groups)
    n_splits = int(min(cv_splits, len(uniq))) if len(uniq) >= 2 else 0

    info = {"n_pairs": int(len(Y)), "poly_degree": poly_degree, "alpha": alpha}
    if n_splits >= 2:
        gkf = GroupKFold(n_splits=n_splits)
        scores = cross_val_score(pipe, X, Y, groups=groups, cv=gkf, scoring="neg_mean_absolute_error")
        info["cv_MAE(mean)"] = float(-scores.mean())
        info["cv_MAE(std)"]  = float(scores.std())
    else:
        info["cv_MAE(mean)"] = None
        info["cv_MAE(std)"]  = None

    pipe.fit(X, Y)
    return pipe, info


# ---------------------------
# 3) Predict helper (returns both x,y next-step)
# ---------------------------
def predict_next_xy(model: Pipeline, X: np.ndarray) -> np.ndarray:
    if X.size == 0:
        return np.zeros((0, 2), float)
    return model.predict(X)


# ---------------------------
# 4) Evaluation helpers (per-subject CSV optional; report only averages)
# ---------------------------
def _eval_on_df(
    model: Pipeline,
    df: pd.DataFrame,
    *,
    hop: int,
    use_dt: bool = True,
    sid_col: str = "sid",
    age_col: str = "age",
    x_col: str = "x",
    y_col: str = "y",
    landmark_col: Optional[str] = "landmark",
) -> Tuple[Dict[str, float], pd.DataFrame]:
    X, Y, groups, meta = build_nextx_dataset(
        df, use_dt=use_dt, hop=hop,
        sid_col=sid_col, age_col=age_col, x_col=x_col, y_col=y_col, landmark_col=landmark_col
    )
    if len(Y) == 0:
        avg = {"mae_x": np.nan, "mae_y": np.nan, "mae_2d": np.nan, "n_pairs": 0}
        return avg, pd.DataFrame(columns=["sid", "n_pairs", "mae_x", "mae_y", "mae_2d"])

    Y_hat = predict_next_xy(model, X)
    err_x  = np.abs(Y_hat[:, 0] - Y[:, 0])
    err_y  = np.abs(Y_hat[:, 1] - Y[:, 1])
    err_2d = np.sqrt((Y_hat[:, 0] - Y[:, 0])**2 + (Y_hat[:, 1] - Y[:, 1])**2)

    avg = {
        "mae_x": float(err_x.mean()),
        "mae_y": float(err_y.mean()),
        "mae_2d": float(err_2d.mean()),
        "n_pairs": int(len(Y)),
    }

    meta_err = meta.copy()
    meta_err["abs_err_x"] = err_x
    meta_err["abs_err_y"] = err_y
    meta_err["err_2d"]    = err_2d

    group_cols = ["sid"]  # per-SID as requested
    per = (meta_err
           .groupby(group_cols, dropna=False)
           .agg(n_pairs=("abs_err_x","size"),
                mae_x=("abs_err_x","mean"),
                mae_y=("abs_err_y","mean"),
                mae_2d=("err_2d","mean"))
           .reset_index())

    return avg, per


def eval_test_mae(
    model: Pipeline,
    df_test: pd.DataFrame,
    *,
    use_dt: bool = True,
    sid_col: str = "sid",
    age_col: str = "age",
    x_col: str = "x",
    y_col: str = "y",
    landmark_col: Optional[str] = "landmark",
) -> Tuple[Dict[str, float], pd.DataFrame]:
    """hop=1 on the SAME test set."""
    return _eval_on_df(
        model, df_test, hop=1, use_dt=use_dt,
        sid_col=sid_col, age_col=age_col, x_col=x_col, y_col=y_col, landmark_col=landmark_col
    )


def eval_skip1_mae(
    model: Pipeline,
    df_test: pd.DataFrame,
    *,
    use_dt: bool = True,
    sid_col: str = "sid",
    age_col: str = "age",
    x_col: str = "x",
    y_col: str = "y",
    landmark_col: Optional[str] = "landmark",
) -> Tuple[Dict[str, float], pd.DataFrame]:
    """hop=2 (skip-1) on the SAME test set."""
    return _eval_on_df(
        model, df_test, hop=2, use_dt=use_dt,
        sid_col=sid_col, age_col=age_col, x_col=x_col, y_col=y_col, landmark_col=landmark_col
    )

In [ ]:
import os
def save_model(model: Pipeline, path: str, info: Optional[Dict[str, Any]] = None) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump({"model": model, "info": info}, f)

In [24]:
import os
import pandas as pd

def run_xy_pipeline(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    subset_name: str,                          # e.g., "male" | "female" | "both"
    *,
    landmarks=None,
    models_dir: str = "./models",
    reports_dir: str = "./reports",
    alpha: float = 1.0,
    use_dt: bool = True,
    poly_degrees=(1, 2),                       # (1 = LinearRidge, >1 = PolyRidge)
    landmark_col: str = "landmark",
    sid_col: str = "sid",
    age_col: str = "age",
    x_col: str = "x",
    y_col: str = "y",
):
    """
    Trains joint (x,y) next-step models per landmark and writes:
      - {reports_dir}/{subset}_nextxy_report_TEST.csv
      - {reports_dir}/{subset}_nextxy_per_sid_TEST.csv
    Saves models to:
      - {models_dir}/{landmark}_{subset}_xy_poly{deg}.pkl
    Returns: (report_df, per_sid_df)
    """
    os.makedirs(models_dir, exist_ok=True)
    os.makedirs(reports_dir, exist_ok=True)

    # Landmarks list (default = present in train_df)
    if landmarks is None:
        landmarks = list(train_df[landmark_col].dropna().unique())

    report_rows = []
    per_sid_chunks = []

    def _label(deg: int) -> str:
        return "LinearRidge" if deg == 1 else f"PolyRidge(d{deg})"

    for l in landmarks:
        tr = train_df[train_df[landmark_col] == l]
        te = test_df[test_df[landmark_col] == l]

        # Build training pairs (t -> t+1), joint xy
        Xtr, Ytr, Gtr, _ = build_nextx_dataset(
            tr, use_dt=use_dt, hop=1,
            sid_col=sid_col, age_col=age_col,
            x_col=x_col, y_col=y_col, landmark_col=landmark_col
        )

        for deg in poly_degrees:
            model, info = fit_nextx_regressor(
                Xtr, Ytr, Gtr,
                alpha=alpha, poly_degree=deg
            )

            # Save model
            model_path = os.path.join(models_dir, f"{l}_{subset_name}_xy_poly{deg}.pkl")
            save_model(model, model_path, info)

            # Evaluate on test (hop=1) and skip-1 (hop=2)
            avg_1,  per_1  = eval_test_mae(
                model, te, use_dt=use_dt,
                sid_col=sid_col, age_col=age_col,
                x_col=x_col, y_col=y_col, landmark_col=landmark_col
            )
            avg_2,  per_2  = eval_skip1_mae(
                model, te, use_dt=use_dt,
                sid_col=sid_col, age_col=age_col,
                x_col=x_col, y_col=y_col, landmark_col=landmark_col
            )

            # ---- rows for averages
            def mkrow(hop, avg):
                return {
                    "subset": subset_name,
                    "landmark": l,
                    "model": _label(deg),
                    "hop": hop,
                    "test_MAE_x": avg["mae_x"],
                    "test_MAE_y": avg["mae_y"],
                    "test_MAE_2D": avg["mae_2d"],
                    "n_pairs": avg["n_pairs"],
                }
            report_rows.extend([mkrow(1, avg_1), mkrow(2, avg_2)])

            # ---- concat per-SID for this landmark/model/hop
            def tag(df_per, hop):
                df = df_per.copy()
                df["subset"]   = subset_name
                df["landmark"] = l
                df["model"]    = _label(deg)
                df["hop"]      = hop
                return df

            per_sid_chunks += [tag(per_1, 1), tag(per_2, 2)]

    # Write outputs
    report_df  = pd.DataFrame(report_rows)
    per_sid_df = pd.concat(per_sid_chunks, ignore_index=True) if per_sid_chunks else pd.DataFrame()

    report_csv  = os.path.join(reports_dir, f"{subset_name}_nextxy_report_TEST.csv")
    per_sid_csv = os.path.join(reports_dir, f"{subset_name}_nextxy_per_sid_TEST.csv")
    report_df.to_csv(report_csv, index=False)
    per_sid_df.to_csv(per_sid_csv, index=False)

    return report_df, per_sid_df


In [25]:
data = pd.read_csv("/data/all_landmark_series_long.csv")
train_sids, test_sids, meta = load_sid_split("/data/splits/sid_split_v1.json")
re_splits = apply_sid_split(data, train_sids, test_sids)

train_all, test_all = re_splits["all"]
train_m, test_m     = re_splits["male"]
train_f, test_f     = re_splits["female"]

In [31]:
landmarks = ['sella','nasion','porion','orbitale','u i apex','point a','u i edge','l i edge','point b','l i apex','pogonion','menton','u 6 apex','u 6 cusp','l 6 cusp','l 6 apex','gonion l','gonion u','condyle','pns','basion','u_6_mcp','l_6_mcp','ans','articular','mid gonion']

In [32]:
# Example calls:
run_xy_pipeline(train_m,   test_m,   "male",   landmarks=landmarks)
run_xy_pipeline(train_f,   test_f,   "female", landmarks=landmarks)
run_xy_pipeline(train_all, test_all, "both",   landmarks=landmarks)


(    subset    landmark          model  hop  test_MAE_x  test_MAE_y  \
 0     both       sella    LinearRidge    1    0.000000    0.000000   
 1     both       sella    LinearRidge    2    0.000000    0.000000   
 2     both       sella  PolyRidge(d2)    1    0.000000    0.000000   
 3     both       sella  PolyRidge(d2)    2    0.000000    0.000000   
 4     both      nasion    LinearRidge    1    0.842886    0.000000   
 ..     ...         ...            ...  ...         ...         ...   
 99    both   articular  PolyRidge(d2)    2    1.255802    1.350558   
 100   both  mid gonion    LinearRidge    1    1.529122    1.275821   
 101   both  mid gonion    LinearRidge    2    2.057436    1.803359   
 102   both  mid gonion  PolyRidge(d2)    1    1.526957    1.292261   
 103   both  mid gonion  PolyRidge(d2)    2    1.988034    1.823560   
 
      test_MAE_2D  n_pairs  
 0       0.000000      314  
 1       0.000000      269  
 2       0.000000      314  
 3       0.000000      269  
 

In [33]:
reports = ["female_nextxy_report_TEST.csv", "male_nextxy_report_TEST.csv", "both_nextxy_report_TEST.csv"]

In [35]:
data = pd.read_csv("reports/"+reports[0])

In [37]:
data

,subset,landmark,model,hop,test_MAE_x,test_MAE_y,test_MAE_2D,n_pairs
0,female,sella,LinearRidge,1,0.000000,0.000000,0.000000,160
1,female,sella,LinearRidge,2,0.000000,0.000000,0.000000,136
2,female,sella,PolyRidge(d2),1,0.000000,0.000000,0.000000,160
3,female,sella,PolyRidge(d2),2,0.000000,0.000000,0.000000,136
4,female,nasion,LinearRidge,1,0.813897,0.000000,0.813897,160
...,...,...,...,...,...,...,...,...
99,female,articular,PolyRidge(d2),2,1.181481,1.082609,1.789480,135
100,female,mid gonion,LinearRidge,1,1.561786,1.169553,2.132847,60
101,female,mid gonion,LinearRidge,2,1.955220,1.549367,2.691927,51
102,female,mid gonion,PolyRidge(d2),1,1.589282,1.175321,2.159707,60


In [41]:
import pandas as pd
import numpy as np

def make_clean_summary(
    report_csv_path: str,
    save_csv: str = None,
    subset: str = None,
    decimals: int = 4,   # <— rounding precision
) -> pd.DataFrame:
    """
    Create a wide per-landmark summary from your average report CSV and round metrics.

    Input CSV must have columns:
      subset, landmark, model, hop, test_MAE_x, test_MAE_y, test_MAE_2D, n_pairs

    Output columns:
      landmark,
      linearridge_test_MAE_x, linearridge_test_MAE_y, linearridge_test_MAE_2D,
      linearridge_skip_test_MAE_x, linearridge_skip_test_MAE_y, linearridge_skip_test_MAE_2D,
      pollyridge_test_MAE_x, pollyridge_test_MAE_y, pollyridge_test_MAE_2D,
      pollyridge_skip_test_MAE_x, pollyridge_skip_test_MAE_y, pollyridge_skip_test_MAE_2D,
      n_pairs, n_pairs_skip

    All float metrics are rounded to `decimals` places; n_pairs columns remain integers.
    """
    df = pd.read_csv(report_csv_path)

    # Normalize column names (strip, drop leading '#')
    df = df.rename(columns={c: c.strip().lstrip("#").strip() for c in df.columns})

    required = {"landmark", "model", "hop", "test_MAE_x", "test_MAE_y", "test_MAE_2D", "n_pairs"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {report_csv_path}: {sorted(missing)}")

    # Optional subset filter if present
    if subset is not None and "subset" in df.columns:
        df = df[df["subset"].astype(str).str.lower() == subset.lower()].copy()

    # Standardize model labels
    model_norm = (
        df["model"].astype(str).str.strip().str.lower()
          .replace({
              "linearridge": "linearridge",
              "poly-ridge": "pollyridge",
              "polyridge": "pollyridge",
              "polyridge(d2)": "pollyridge",
              "poly ridge(d2)": "pollyridge",
              "poly ridge": "pollyridge",
          })
    )
    df = df.assign(model_norm=model_norm)
    df = df[df["model_norm"].isin(["linearridge", "pollyridge"])].copy()

    rows = []
    for landmark, g in df.groupby("landmark", sort=False):
        row = {"landmark": landmark}

        # n_pairs per hop (take from any model row at that hop)
        hop1 = g[g["hop"] == 1]
        hop2 = g[g["hop"] == 2]
        row["n_pairs"]      = int(hop1["n_pairs"].dropna().iloc[0]) if not hop1.empty else np.nan
        row["n_pairs_skip"] = int(hop2["n_pairs"].dropna().iloc[0]) if not hop2.empty else np.nan

        def fill(model_key: str, hop: int, alias: str, suffix: str):
            sel = g[(g["model_norm"] == model_key) & (g["hop"] == hop)]
            if sel.empty:
                row[f"{alias}{suffix}_test_MAE_x"]  = np.nan
                row[f"{alias}{suffix}_test_MAE_y"]  = np.nan
                row[f"{alias}{suffix}_test_MAE_2D"] = np.nan
            else:
                rec = sel.iloc[0]
                row[f"{alias}{suffix}_test_MAE_x"]  = float(rec["test_MAE_x"])
                row[f"{alias}{suffix}_test_MAE_y"]  = float(rec["test_MAE_y"])
                row[f"{alias}{suffix}_test_MAE_2D"] = float(rec["test_MAE_2D"])

        # LinearRidge: hop=1 & hop=2
        fill("linearridge", 1, "linearridge", "")
        fill("linearridge", 2, "linearridge", "_skip")

        # PolyRidge(d2) => 'pollyridge': hop=1 & hop=2
        fill("pollyridge", 1, "pollyridge", "")
        fill("pollyridge", 2, "pollyridge", "_skip")

        rows.append(row)

    out = pd.DataFrame(rows)
    # Preferred column order
    cols = ["landmark",
            "linearridge_test_MAE_x", "linearridge_test_MAE_y", "linearridge_test_MAE_2D",
            "linearridge_skip_test_MAE_x", "linearridge_skip_test_MAE_y", "linearridge_skip_test_MAE_2D",
            "pollyridge_test_MAE_x", "pollyridge_test_MAE_y", "pollyridge_test_MAE_2D",
            "pollyridge_skip_test_MAE_x", "pollyridge_skip_test_MAE_y", "pollyridge_skip_test_MAE_2D",
            "n_pairs", "n_pairs_skip"]
    out = out.reindex(columns=cols)

    # Make pair-count columns integer (nullable) and round only float metrics
    for c in ["n_pairs", "n_pairs_skip"]:
        if c in out.columns:
            out[c] = out[c].astype("Int64")

    float_cols = out.select_dtypes(include=["float32", "float64"]).columns
    out[float_cols] = out[float_cols].round(decimals)

    if save_csv:
        out.to_csv(save_csv, index=False)

    return out


In [45]:
# Example:
clean_df = make_clean_summary("./reports/both_nextxy_report_TEST.csv",
                              save_csv="./reports/both_clean_summary.csv",
                              subset="both")   # omit subset=... if not needed
print(clean_df.head())


   landmark  linearridge_test_MAE_x  linearridge_test_MAE_y  \
0     sella                  0.0000                  0.0000   
1    nasion                  0.8429                  0.0000   
2    porion                  1.9435                  2.4122   
3  orbitale                  1.1920                  1.1209   
4  u i apex                  1.1871                  1.5162   

   linearridge_test_MAE_2D  linearridge_skip_test_MAE_x  \
0                   0.0000                       0.0000   
1                   0.8429                       1.1112   
2                   3.4247                       2.0428   
3                   1.8470                       1.3937   
4                   2.1369                       1.5334   

   linearridge_skip_test_MAE_y  linearridge_skip_test_MAE_2D  \
0                       0.0000                        0.0000   
1                       0.0000                        1.1112   
2                       2.5718                        3.6103   
3         